# Module 9 — Experiment Tracking with MLflow (Preview)
## Chapter 2 · Computer Vision Learning

**⚠️ Run this notebook on Google Colab with T4 GPU enabled**

This notebook is a **preview of MLflow**: tracking the same transfer-learning experiment
from Module 5 (ResNet50 on the flower dataset) using MLflow's local file-based tracking
store — no external server required.

Every run records:
- Dataset version, model, epochs, learning rate, batch size, optimizer, scheduler (as MLflow params)
- Per-epoch metrics (loss, accuracy), logged as MLflow metrics
- Training time (logged as a metric)
- Hardware (logged as a tag)
- The trained model itself, logged as an MLflow artifact


## Setup & GPU Check

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'GPU Available: {torch.cuda.is_available()}')
print(f'GPU Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB' if torch.cuda.is_available() else '')

if not torch.cuda.is_available():
    print('\n⚠️ WARNING: No GPU found! Please enable GPU in Colab:')
    print('   Runtime → Change runtime type → GPU (T4 recommended)')


## Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted')


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mlflow'])
print('✓ mlflow installed')


## Import Libraries

In [ ]:
import os, warnings, time, platform, subprocess
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import torchvision.models as models

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('✓ All imports successful')


In [ ]:
import mlflow
import mlflow.pytorch


## Configure Paths & Parameters

In [ ]:
DATASET_BASE = Path('/content/drive/MyDrive/Data Science/Computer Vision/learning-assignments/datasets/flowers')
TRAIN_DIR = DATASET_BASE / 'splits' / 'train'
VAL_DIR = DATASET_BASE / 'splits' / 'val'
TEST_DIR = DATASET_BASE / 'splits' / 'test'

OUTPUT_DIR = Path('/content/drive/MyDrive/Data Science/Computer Vision/learning-assignments/flower_classifier_experiment_tracking_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Train dir exists: {TRAIN_DIR.exists()}')
print(f'Output dir: {OUTPUT_DIR}')

# Dataset version: names the exact split this run trained on (record this alongside metrics)
DATASET_VERSION = 'flowers-splits-v1'

IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE, NUM_CLASSES = 32, 3
LEARNING_RATE, EPOCHS = 1e-3, 15
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
MODEL_NAME = 'resnet50'
OPTIMIZER_NAME = 'Adam'
SCHEDULER_NAME = 'ReduceLROnPlateau'

print('✓ Configuration loaded')


## Configure MLflow Tracking Store

In [ ]:
MLFLOW_TRACKING_DIR = OUTPUT_DIR / 'mlruns'
mlflow.set_tracking_uri(f'file:{MLFLOW_TRACKING_DIR}')
mlflow.set_experiment('flower-classifier-transfer-learning')
print(f'✓ MLflow tracking to {MLFLOW_TRACKING_DIR}')


## Load Data

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

train_dataset = ImageFolder(root=TRAIN_DIR, transform=train_transform)
val_dataset = ImageFolder(root=VAL_DIR, transform=val_transform)
test_dataset = ImageFolder(root=TEST_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
print(f'Classes: {class_names}')
print(f'Samples - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')
print('✓ Data loaded')


## Transfer Learning Model Wrapper

In [ ]:
class TransferLearningModel(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        num_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        if features.dim() > 2:
            features = features.mean(dim=[2, 3])
        return self.classifier(features)

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True

print('✓ TransferLearningModel defined')


## Hardware Info Helper

In [ ]:
def get_hardware_info():
    info = {
        'device': str(device),
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'gpu_count': torch.cuda.device_count(),
        'cpu': platform.processor() or platform.machine(),
        'torch_version': torch.__version__,
    }
    return info

hardware_info = get_hardware_info()
print(hardware_info)


## Training Functions (log every epoch to MLflow)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        l = criterion(outputs, labels)
        l.backward()
        optimizer.step()
        loss += l.item()
        _, pred = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()
    return loss / len(loader), correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            l = criterion(outputs, labels)
            loss += l.item()
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    return loss / len(loader), correct / total

def run_experiment(run_name, freeze, epochs=EPOCHS, lr=LEARNING_RATE):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            'dataset_version': DATASET_VERSION,
            'model': MODEL_NAME,
            'freeze_backbone': freeze,
            'epochs': epochs,
            'learning_rate': lr,
            'batch_size': BATCH_SIZE,
            'optimizer': OPTIMIZER_NAME,
            'scheduler': SCHEDULER_NAME,
        })
        mlflow.set_tag('hardware', hardware_info['gpu_name'])

        backbone = models.resnet50(pretrained=True)
        model = TransferLearningModel(backbone, NUM_CLASSES).to(device)
        if freeze:
            model.freeze_backbone()
        else:
            model.unfreeze_backbone()

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

        start_time = time.time()
        best_val_acc = 0.0
        for epoch in range(epochs):
            tl, ta = train_epoch(model, train_loader, criterion, optimizer, device)
            vl, va = val_epoch(model, val_loader, criterion, device)
            scheduler.step(vl)
            best_val_acc = max(best_val_acc, va)

            mlflow.log_metrics({
                'train_loss': tl, 'train_acc': ta,
                'val_loss': vl, 'val_acc': va,
                'learning_rate': optimizer.param_groups[0]['lr'],
            }, step=epoch)

            print(f'  [{run_name}] Epoch {epoch+1:2d}/{epochs} | TL: {tl:.4f} TA: {ta:.4f} | VL: {vl:.4f} VA: {va:.4f}')

        training_time = time.time() - start_time

        model.eval()
        test_correct = 0
        with torch.no_grad():
            for images, labels in test_loader:
                outputs = model(images.to(device))
                test_correct += (torch.max(outputs, 1)[1] == labels.to(device)).sum().item()
        test_acc = test_correct / len(test_dataset)

        mlflow.log_metric('best_val_acc', best_val_acc)
        mlflow.log_metric('test_acc', test_acc)
        mlflow.log_metric('training_time_sec', training_time)
        mlflow.pytorch.log_model(model, artifact_path='model')

        print(f'  ✓ {run_name} done in {training_time:.1f}s | best val acc {best_val_acc:.4f} | test acc {test_acc:.4f}')
        return {'run_name': run_name, 'test_acc': test_acc, 'best_val_acc': best_val_acc, 'training_time': training_time}

print('✓ Training functions defined')


## Run Experiments: Frozen Backbone vs Fine-Tuned

In [ ]:
results = []
results.append(run_experiment('resnet50_frozen', freeze=True, lr=LEARNING_RATE))
results.append(run_experiment('resnet50_finetuned', freeze=False, lr=LEARNING_RATE / 10))

results_df = pd.DataFrame(results)
results_df


## Query Runs from MLflow

In [ ]:
runs_df = mlflow.search_runs(experiment_names=['flower-classifier-transfer-learning'])
runs_df[['tags.mlflow.runName', 'params.model', 'params.freeze_backbone', 'metrics.test_acc', 'metrics.best_val_acc', 'metrics.training_time_sec']]


## View the MLflow UI

In [ ]:
print('Run this in a terminal (or a separate Colab cell with `!`) to browse runs locally:')
print(f'mlflow ui --backend-store-uri file:{MLFLOW_TRACKING_DIR} --port 5000')


## Summary

In [ ]:
print('='*70)
print('EXPERIMENT TRACKING SUMMARY — MLflow')
print('='*70)
for r in results:
    print(f"{r['run_name']:20s} | val_acc={r['best_val_acc']:.4f} | test_acc={r['test_acc']:.4f} | time={r['training_time']:.1f}s")
print(f"\nDataset version : {DATASET_VERSION}")
print(f"Hardware        : {hardware_info['gpu_name']}")
print(f"Tracking store  : {MLFLOW_TRACKING_DIR}")
